# AI Lead Desk — LLM and Admissions Counselor

## Objective
Build the actual AI admissions counselor using LangChain
and an API-based Large Language Model (LLM).

## Responsibilities
- Connect to the hosted LLM securely.
- Conduct natural admissions conversations.
- Follow institute-approved counselling rules.
- Use conversation history to maintain context.
- Prepare for integration with the institute knowledge base (RAG).
- Support English, Hindi, Marathi and mixed-language conversations.

## Current stage
LLM connection and conversational behaviour.

## Later stages
Knowledge retrieval, student profiling, course recommendations,
objection detection and integration with the lead management system.

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load API credentials from the project's .env file, if present.
load_dotenv()

# Check whether the API key is available without displaying it.
if os.getenv("GROQ_API_KEY"):
    print("Groq API key found.")
else:
    print("Groq API key not found. Configure GROQ_API_KEY in your .env file.")

Groq API key found.


In [3]:
# Initialize the Groq-hosted LLM through LangChain.

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_tokens=512,
    max_retries=2
)

# Test the actual model connection.
response = llm.invoke(
    "Introduce yourself as an AI admissions counselor in one sentence."
)

print("Model initialized:", llm.model_name)
print("\nAI response:")
print(response.content)

Model initialized: openai/gpt-oss-120b

AI response:
I’m an AI admissions counselor here to guide you through every step of the application process with personalized, data‑driven advice.


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage

COUNSELOR_SYSTEM_PROMPT = """
You are the AI admissions counselor for an educational institute.

Your responsibilities:
1. Have natural, helpful conversations with prospective students.
2. Understand their background, education, skills, goals and constraints.
3. Support English, Hindi, Marathi and mixed-language conversations.
4. Ask relevant follow-up questions instead of interrogating the student.
5. Explain suitable next steps clearly and politely.

Accuracy rules:
- Never invent institute fees, discounts, schedules or placement guarantees.
- Use verified institute information only when answering institute-specific questions.
- If the required information is missing, explain that it cannot be verified
  and offer to connect the student to a human counselor.
- Never guarantee admission, employment or career outcomes.
- Keep responses concise and conversational.

Available verified institute information:
{institute_context}
"""


def counsel_once(student_message: str, institute_context: str = ""):
    """Generate a genuine counselor response using the configured LLM."""

    context = institute_context.strip()

    if not context:
        context = "No verified institute information has been provided yet."

    system_prompt = COUNSELOR_SYSTEM_PROMPT.format(
        institute_context=context
    )

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=student_message)
    ]

    return llm.invoke(messages)


# Test an actual counselling conversation
student_message = "मला IT मध्ये career करायचं आहे. मी कुठून सुरुवात करू?"

response = counsel_once(student_message)

print("Student:", student_message)
print("\nAI Counselor:", response.content)

Student: मला IT मध्ये career करायचं आहे. मी कुठून सुरुवात करू?

AI Counselor: नमस्कार! IT‑कॅरिअर सुरू करण्यासाठी काही महत्त्वाचे पायऱ्या आहेत, पण त्यापैकी कोणत्या तुमच्यासाठी योग्य आहेत हे जाणून घेण्यासाठी थोडी माहिती हवी आहे.

1. **शिक्षणाचा स्तर** – सध्या तुम्ही कोणत्या वर्ग/कॉलेजमध्ये आहात? (उदा. 12वी, बॅचलर, कामानंतरचे कोर्स इ.)

2. **आवडती क्षेत्रे** – IT मध्ये तुम्हाला कोणत्या भागात जास्त रस आहे?  
   - सॉफ्टवेअर डेव्हलपमेंट (कोडिंग)  
   - वेब/मोबाइल अॅप विकास  
   - डेटा सायन्स / मशीन लर्निंग  
   - नेटवर्किंग / सायबर‑सुरक्षा  
   - क्लाउड / DevOps इ.

3. **आधीचा अनुभव** – तुम्ही काही प्रोग्रॅमिंग भाषा (जसे Python, Java, C++) किंवा इतर तांत्रिक कौशल्ये आधीच शिकली आहेत का?

4. **वेळ व बजेट** – तुम्ही पूर्णवेळ अभ्यास करु शकता का, की अर्धवेळ/ऑनलाइन कोर्सेस पसंत कराल? तसेच आर्थिक मर्यादा असल्यास त्याबद्दलही कळवू शकाल.

5. **भौगोलिक मर्यादा** – तुम्ही कोणत्या शहरात किंवा प्रदेशात राहता? काही संस्थांचे कॅम्पस किंवा प्रशिक्षण केंद्रे जवळ आहेत का, हे


## Conversation Memory

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Store conversation histories separately for each student/session.
conversation_history = {}


def counsel_chat(
    conversation_id: str,
    student_message: str,
    institute_context: str = ""
):
    """Generate an AI counselor reply while preserving conversation history."""

    system_prompt = COUNSELOR_SYSTEM_PROMPT.format(
        institute_context=(
            institute_context.strip()
            or "No verified institute information has been provided yet."
        )
    )

    # Initialize memory for a new conversation.
    if conversation_id not in conversation_history:
        conversation_history[conversation_id] = [
            SystemMessage(content=system_prompt)
        ]
    else:
        # Refresh the system instructions/context for this turn.
        conversation_history[conversation_id][0] = SystemMessage(
            content=system_prompt
        )

    messages = conversation_history[conversation_id]

    # Add the student's latest message.
    messages.append(HumanMessage(content=student_message))

    try:
        response = llm.invoke(messages)
    except Exception:
        # Remove the unanswered user message if the model request fails.
        messages.pop()
        raise

    # Save the AI response so future turns have context.
    messages.append(AIMessage(content=response.content))

    return response.content


# Test a second turn in the same conversation.
conversation_id = "rahul_demo"

reply = counsel_chat(
    conversation_id,
    "मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही."
)

print("Student:", "मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही.")
print("\nAI Counselor:", reply)
print("\nMessages in conversation:", len(conversation_history[conversation_id]) - 1)

Student: मी सध्या कॉलेजच्या पहिल्या वर्षात आहे आणि मला coding चा अनुभव नाही.

AI Counselor: नमस्कार! कॉलेजच्या पहिल्या वर्षात असताना कोडिंगचा अनुभव नसणे पूर्णपणे सामान्य आहे—बऱ्याच विद्यार्थ्यांनाही नंतरच सुरुवात करावी लागते. मी तुम्हाला योग्य मार्गदर्शन देण्यासाठी काही छोटे प्रश्न विचारतो, ज्यामुळे तुमची आवड, उद्दिष्टे आणि उपलब्ध वेळ यांचा चांगला आढावा मिळेल.

1. **तुम्हाला कोणत्या क्षेत्रात करिअर करायचे आहे?**  
   - सॉफ्टवेअर विकास (वेब, मोबाइल, गेम), डेटा सायन्स, AI/ML, इत्यादी.  
   - किंवा अजून ठरलेलं नाही, फक्त कोडिंग शिकून बघायचे आहे?

2. **कोणत्या प्रकारचा कोडिंग तुम्हाला अधिक आकर्षक वाटतो?**  
   - वेब (HTML, CSS, JavaScript)  
   - मोबाइल अॅप (Android – Kotlin/Java, iOS – Swift)  
   - डेटा/विश्लेषण (Python, R)  
   - इतर (उदा. गेम डेव्हलपमेंट, एम्बेडेड सिस्टम)

3. **तुम्ही दर आठवड्यात कोडिंगसाठी किती वेळ देऊ शकता?**  
   - 5‑10 तास, 10‑15 तास, इत्यादी.

4. **तुम्हाला इंग्रजी किंवा मराठी/हिंदी मध्ये शिकणे सोयीचे वाटते?**  
   - काही कोर्सेस दोन्ही भाषेत उपलब्ध असू शकतात, त्य

In [6]:
from langchain_groq import ChatGroq

# Reinitialize the same model with a larger response budget.
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_tokens=1024,
    max_retries=2
)

print("Model:", llm.model_name)
print("Maximum response tokens: 1024")
print("Conversation function will use this updated model.")

Model: openai/gpt-oss-120b
Maximum response tokens: 1024
Conversation function will use this updated model.


In [7]:
# Continue the same conversation created in Cell 5.
student_message = (
    "मला coding सुरू करायचं आहे. माझ्यासाठी सुरुवात कशी करू?"
)

reply = counsel_chat(
    conversation_id="rahul_demo",
    student_message=student_message
)

print("Student:", student_message)
print("\nAI Counselor:", reply)
print("\nConversation messages:",
      len(conversation_history["rahul_demo"]) - 1)

Student: मला coding सुरू करायचं आहे. माझ्यासाठी सुरुवात कशी करू?

AI Counselor: कोडिंग सुरू करायची इच्छा खूप छान आहे! खाली काही सोपे पायऱ्या दिल्या आहेत ज्या तुम्हाला पायाभरणीपासून पुढे जाण्यास मदत करतील.

---

### १. उद्दिष्ट ठरवा  
- **कोणत्या प्रकारचा सॉफ्टवेअर बनवायचा** आहे? (वेब साइट, मोबाइल अॅप, डेटा विश्लेषण, गेम इ.)  
- तुमचा पहिला प्रोजेक्ट छोटा आणि व्यावहारिक असावा – जसे की “टू‑डू लिस्ट” अॅप किंवा साधी वेबसाइट.

### २. भाषा निवडा  
| उद्दिष्ट | शिफारस केलेली भाषा | का? |
|----------|-------------------|------|
| वेब (फ्रंट‑एंड) | **HTML + CSS + JavaScript** | ब्राउझरमध्ये थेट चालते, लवकर परिणाम दिसतो |
| वेब (बॅक‑एंड) | **Python** (Flask/Django) किंवा **Node.js** (JavaScript) | सोपी सिंटॅक्स, बरीच ट्युटोरियल्स |
| मोबाइल अॅप | **Kotlin** (Android) किंवा **Swift** (iOS) | प्लॅटफॉर्म‑विशिष्ट, नोकरीच्या संधी जास्त |
| डेटा/AI | **Python** (Pandas, NumPy, Scikit‑learn) | डेटा सायन्स आणि मशीन लर्निंगचे मानक |

**सुरुवातीला** मी **Python** किंवा **JavaScript** शिफारस करतो – दोन्ही 